# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeref538/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I read *The State of AI-Driven SEO in Numbers* (FlyRank, March 2026) and picked two findings.
I am not grading the paper. It says plainly on page 36 that it is an **observational study**,
that ML pages are **exploratory appendix material**, and that it reports **no p-values or
confidence intervals**. Those are disclosures most public reports skip. My questions are the
ones I would want asked of my own notebook — and, as it turns out, two of them apply to my
Week-5 model too.

---

### Finding #1 (page 6, tagged CONFIRMED) — "The Anatomy of Growing Content"

**What it says.** Growing pages are longer and younger than declining ones: 3.2K vs 2.3K words
(+37.6%), 184 vs 230 days old. Sample is large — 74,187 rising against 45,272 falling.

**Where does the label come from?** `trend_direction` is defined on page 5 as the change in
impressions over the last 30 days versus the 30 before that (up = more than +10%, down = more
than -10%). Word count and age are read from the same snapshot as that label. So the page's
length is measured *after* the window that decided whether it was growing — the comparison is a
photograph, not a film.

**The question I would ask.** Does the gap survive holding age fixed? The paper names content
age as a confounder on page 36, for the AI-model comparison. The same confounder sits inside
this finding: newer pages are written to newer length norms *and* newer pages are the ones with
room to grow off a low base. A page published last month can post +70% impressions from a small
number, while a two-year-old page holding steady gets tagged "down" on a much larger base. Age
could be producing both halves of the pattern. A within-age-band comparison — the 0-90 day
cohort against itself, then 90-180, and so on — would separate "longer content grows" from
"younger content grows and younger content is longer."

**Why this matters practically.** The finding is worded carefully ("directionally robust",
"observational comparison"). But the *What to Do* box next to it says **"Expand thin pages that
already earn impressions"**, and that is a causal instruction taken from a correlational gap.
If the real driver is age, a team could add 900 words to old pages and see nothing move. My
suggestion would be to keep the finding as-is and soften the action to a testable one — pick 20
pages, expand 10, hold 10 back, compare after 60 days. The paper's own *Measure* line already
asks for a 30/60-day follow-up, so it is one step from being a real experiment.

---

### Finding #2 (page 29, ML appendix) — "What Predicts Growth?"

**What it says.** A logistic regression with **71% holdout accuracy** separating growing from
declining pages, with content age as the strongest negative signal.

**Does the validation design support the claim?** Two things I would want stated next to that
71%.

**(a) 71% against what?** Accuracy only means something beside the rate you would get by
guessing the majority class every time. The paper gives me the counts to work this out from its
own page 6 — and I do that in the cell below. It lands at **62.1% growing**, so a model that
answered "growing" for every single page would already score about 62%. The 71% is roughly
**nine points of skill**, not seventy-one. That is a real result, and it reads very differently
without the baseline beside it.

**(b) Was the holdout grouped by brand?** Page 36 says the split is **80/20**, over 61.8K pages
drawn from 57 brands. It does not say whether pages from one brand can land on both sides. If
they can, some of that 71% may be the model recognising a brand rather than recognising growth —
pages from one site share templates, domain strength, publishing cadence, and topic. The fix is
small: `GroupKFold(groups=brand_id)` instead of a plain split, and report both numbers. **I ran
exactly this test on my own model in section 2, and the honest number was 0.080 AUC lower.**
That is why I am raising it — not because I suspect the paper, but because I found it in my own
work first.

**One more, smaller.** Two of the model's named features are `Days Since Update` and
`Days Visible`. If "days visible" counts days with impressions inside the same 30-day window
that defines the label, then part of the answer is sitting in the inputs. Days visible cannot
rise while impressions fall. A one-line note on which window each feature is measured over
would close that question completely.

In [1]:
# Checking the paper's own numbers against each other, using only what it published.
# Page 6 gives the up/down counts; page 29 gives the model's accuracy. The baseline
# is the thing that turns one into context for the other.
up, down = 74_187, 45_272          # paper, Finding #1, page 6
majority = up / (up + down)
model_acc = 0.71                   # paper, ML appendix, page 29

print(f"growing pages : {up:,}")
print(f"declining     : {down:,}")
print(f"majority class: {majority:.3f}  <- accuracy of a model that always answers 'growing'")
print(f"paper's model : {model_acc:.3f}")
print(f"skill over the naive answer: {model_acc - majority:+.3f}  ({(model_acc-majority)*100:.1f} points)")
print()
print("Not a criticism of the model -- 9 points is a real signal on a hard problem.")
print("It is an argument for printing the base rate next to every accuracy, mine included.")

growing pages : 74,187
declining     : 45,272
majority class: 0.621  <- accuracy of a model that always answers 'growing'
paper's model : 0.710
skill over the naive answer: +0.089  (8.9 points)

Not a criticism of the model -- 9 points is a real signal on a hard problem.
It is an argument for printing the base rate next to every accuracy, mine included.


## 2. My model under an honest split (before/after)

Now the same lens on my own Week-5 work.

My data has 13,562 eligible pages but only **29 clients**. Pages from one client share a lot:
the same site, the same templates, the same publishing rhythm, often the same topics. A plain
random split scatters one client's pages across both sides of the wall, so the model can learn
"this looks like client 7's pages, and client 7 is mostly declining" and score well without
knowing anything about decline.

The picture I keep in my head: it is the difference between an exam with new questions and an
exam where you have already seen half the answer key. Both give you a mark. Only one tells you
whether you learned the subject.

**Before** is a random split on rows. **After** is `GroupShuffleSplit` on `client_id`, which
keeps every page belonging to one client on the same side of the wall. Same model, same
features, same eligibility gate — the only thing that changes is where the line is drawn.

In [2]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# identical eligibility gate to ML-07 and ML-08, so this compares like with like
elig = df[(df.impressions_90d >= 250) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
tier_med = elig.groupby("position_tier")["ctr"].transform("median")
elig["ctr_gap_ratio"] = ((tier_med - elig.ctr) / tier_med.replace(0, np.nan)).clip(lower=0).fillna(0)
elig["freshness_term"] = (elig.days_since_last_update / 365).clip(upper=1)

FEATURES = ["ctr_gap_ratio", "freshness_term", "ctr", "avg_position", "impressions_90d",
            "clicks_90d", "days_since_last_update", "content_age_days", "word_count",
            "engagement_rate", "scroll_rate", "days_with_impressions", "search_volume",
            "competition"]

X = elig[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y = elig.is_declining.values
groups = elig.client_id

print(f"eligible pages: {len(elig):,} | clients: {elig.client_id.nunique()} | base rate: {y.mean():.3f}")


def p_at_50(scores, yy):
    "Share of the top 50 ranked pages that really were declining."
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(yy)[order[:50]].mean())


# metric self-check: of the top 2 ranked, one is positive -> 0.5. If this ever fails,
# every score below it is meaningless, so it runs before anything else.
def p_at_k(scores, yy, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(yy)[order[:k]].mean())
assert p_at_k([9, 8, 7], [1, 0, 1], 2) == 0.5, "metric is wrong, stop here"


def evaluate(tr, te, label):
    m = GradientBoostingClassifier(random_state=42).fit(X.iloc[tr], y[tr])
    s = m.predict_proba(X.iloc[te])[:, 1]
    shared = len(set(elig.iloc[tr].client_id) & set(elig.iloc[te].client_id))
    return dict(split=label,
                test_rows=len(te),
                test_clients=elig.iloc[te].client_id.nunique(),
                clients_on_both_sides=shared,
                base_rate=round(y[te].mean(), 3),
                P_at_50=round(p_at_50(s, y[te]), 3),
                ROC_AUC=round(roc_auc_score(y[te], s), 3),
                PR_AUC=round(average_precision_score(y[te], s), 3)), m, s


tr_r, te_r = next(ShuffleSplit(1, test_size=0.3, random_state=42).split(X, y))
before, _, _ = evaluate(tr_r, te_r, "BEFORE - random rows")

tr_g, te_g = next(GroupShuffleSplit(1, test_size=0.3, random_state=42).split(X, y, groups))
after, gb, s_grouped = evaluate(tr_g, te_g, "AFTER - grouped by client")

print()
print(pd.DataFrame([before, after]).to_string(index=False))

eligible pages: 13,562 | clients: 29 | base rate: 0.610



                    split  test_rows  test_clients  clients_on_both_sides  base_rate  P_at_50  ROC_AUC  PR_AUC
     BEFORE - random rows       4069            27                     27      0.607     0.94    0.728   0.795
AFTER - grouped by client       2955             9                      0      0.551     0.88    0.618   0.667


In [3]:
# One split could be luck. Run the same before/after over 8 different random draws
# and see whether the gap holds its direction.
rows = []
for seed in range(8):
    a, _, _ = evaluate(*next(ShuffleSplit(1, test_size=0.3, random_state=seed).split(X, y)), "random")
    b, _, _ = evaluate(*next(GroupShuffleSplit(1, test_size=0.3, random_state=seed).split(X, y, groups)), "grouped")
    rows.append(dict(seed=seed, random_AUC=a["ROC_AUC"], grouped_AUC=b["ROC_AUC"],
                     gap=round(a["ROC_AUC"] - b["ROC_AUC"], 3)))

gaps = pd.DataFrame(rows)
print(gaps.to_string(index=False))
print()
print(f"mean gap (random - grouped): {gaps.gap.mean():+.3f}")
print(f"random split scored higher in {int((gaps.gap > 0).sum())} of 8 draws")

 seed  random_AUC  grouped_AUC   gap
    0       0.719        0.629 0.090
    1       0.722        0.665 0.057
    2       0.713        0.652 0.061
    3       0.722        0.620 0.102
    4       0.718        0.637 0.081
    5       0.730        0.627 0.103
    6       0.723        0.650 0.073
    7       0.710        0.635 0.075

mean gap (random - grouped): +0.080
random split scored higher in 8 of 8 draws


**What I observe.** The random split reports ROC-AUC **0.728** and Precision@50 of **0.94**.
The grouped split, on the same pages with the same model, reports **0.618** and **0.88**.

The gap is not a one-off. Across 8 different draws the random split scored higher **every
time**, by **0.080 AUC on average** (smallest 0.057, largest 0.103). A difference that never
once flips direction is a property of the split, not noise.

The `clients_on_both_sides` column is the plain evidence for why: **27** under the random split,
**0** under the grouped one. Under the naive split there was effectively no client the model
had not already met.

I am treating **0.618** as my real number. The 0.728 measures how well the model does on
clients it has already seen, which is not the job — the job is a new client's pages arriving
next month. Reporting both, and the gap between them, is more informative than reporting either
alone.

One honest limitation: 9 test clients is a small holdout. The grouped number carries real
uncertainty of its own, which is why section 4 rewrites my claim in directional language rather
than quoting a decimal as if it were exact.

## 3. Leakage audit

Leakage means the answer sneaks into the inputs. The model looks brilliant and has learned
nothing. It is the failure that looks most like success, which is what makes it worth hunting
deliberately rather than hoping.

Two columns in this dataset are computed **from** the label. `trend_direction` *is* the label.
`trend_pct` is the size of the same 30-day impression change. Neither can ever be a feature.

Rather than just assert my feature list is clean, I do what the repo's leakage skill says: add
a leaky column on purpose and check the score confesses. If it does not jump, my test harness
is broken and every number I have reported is suspect.

In [4]:
BANNED = {"trend_direction", "trend_pct", "is_declining"}
leaked = set(FEATURES) & BANNED
print(f"label-derived columns in FEATURES: {leaked if leaked else 'none'}")
assert not leaked, "a label-derived column is in the feature list"

# Window check: every feature must be knowable at prediction time.
# days_with_impressions is the one worth naming out loud -- it counts days the page
# was visible in the same 90-day frame the trend is cut from, so it sits closest to
# the label of anything I kept. Keeping it, flagging it.
print("\nfeature windows:")
print("  90-day aggregates : impressions_90d, clicks_90d, ctr, avg_position, days_with_impressions")
print("  page metadata     : content_age_days, word_count, days_since_last_update")
print("  behaviour         : engagement_rate, scroll_rate")
print("  query context     : search_volume, competition")
print("  rule terms        : ctr_gap_ratio, freshness_term  (built from the columns above)")

# The deliberate-leak test.
X_leaky = X.copy()
X_leaky["trend_pct"] = elig["trend_pct"].fillna(0).values

m_leak = GradientBoostingClassifier(random_state=42).fit(X_leaky.iloc[tr_g], y[tr_g])
s_leak = m_leak.predict_proba(X_leaky.iloc[te_g])[:, 1]
auc_leak = roc_auc_score(y[te_g], s_leak)

imp = pd.Series(m_leak.feature_importances_, index=X_leaky.columns).sort_values(ascending=False)

print(f"\nhonest ROC-AUC (clean features): {after['ROC_AUC']:.3f}")
print(f"ROC-AUC with trend_pct added   : {auc_leak:.3f}")
print(f"top feature in the leaky model : {imp.index[0]}  (importance {imp.iloc[0]:.3f})")
print()
print("A jump to a perfect score, with the leaked column taking ~all the importance, is the")
print("confession we wanted to see. The harness detects leakage, so the 0.618 is trustworthy")
print("as an honest measurement. Dropping trend_pct again for everything that follows.")

label-derived columns in FEATURES: none

feature windows:
  90-day aggregates : impressions_90d, clicks_90d, ctr, avg_position, days_with_impressions
  page metadata     : content_age_days, word_count, days_since_last_update
  behaviour         : engagement_rate, scroll_rate
  query context     : search_volume, competition
  rule terms        : ctr_gap_ratio, freshness_term  (built from the columns above)



honest ROC-AUC (clean features): 0.618
ROC-AUC with trend_pct added   : 1.000
top feature in the leaky model : trend_pct  (importance 0.999)

A jump to a perfect score, with the leaked column taking ~all the importance, is the
confession we wanted to see. The harness detects leakage, so the 0.618 is trustworthy
as an honest measurement. Dropping trend_pct again for everything that follows.


## 4. Claim rewrite

Here is the boldest sentence from my Week-5 notebook, and what is wrong with it.

> **Before:** "Gradient boosting beats the hand-written rule, 0.88 vs 0.86 Precision@50."

Three problems, all mine:

1. **"Beats" is a verdict, and the margin does not support one.** In ML-08 I measured the rule's
   Precision@50 swinging between **0.760 and 0.920** across 500 random tie-breaks, because
   `ctr_gap_ratio` pins at its cap for every zero-click page and leaves large ties. A 0.02
   difference sits inside a 0.16-wide band.
2. **It reports one split.** Over 8 client splits the model won 6 and lost 2, mean margin
   +0.056. "Wins most of the time" is a different sentence from "beats".
3. **It hides what the label is.** `trend_direction` is computed from the same 90-day window as
   the features, so this ranks pages *already labelled* declining. It does not predict decline
   before it happens. Anyone reading "beats" would assume the second thing.

> **After:** "On 9 held-out clients the model never trained on, gradient boosting ranked
> **44 of its top 50** pages as already-declining, against **39 of 50** for the hand-written
> rule (observed, single split). Across 8 client splits the model placed higher in 6, by an
> average of +0.056 Precision@50 — directional, not decisive, since re-running the rule with
> different tie-breaking moves its own score between 0.760 and 0.920. Both methods rank pages
> that are *currently* labelled declining; neither is measured on predicting future decline.
> Decision-support for choosing which 50 pages an editor reviews first, not a forecast."

Longer, and every clause is one I can defend. The cell below is the error evidence behind it —
because a claim about a ranked list should come with a look at what the list gets wrong.

In [5]:
test = elig.iloc[te_g].copy()
test["model_score"] = s_grouped
top50 = test.sort_values("model_score", ascending=False).head(50)

print(f"honest split: model's top 50 -> {int(top50.is_declining.sum())} of 50 were really declining")
print(f"test-set base rate for comparison: {y[te_g].mean():.3f}")
print()

cols = ["ctr", "clicks_90d", "impressions_90d", "avg_position", "days_since_last_update", "ctr_gap_ratio"]
profile = top50.groupby("is_declining")[cols].median().round(3)
profile.index = ["false alarms (not declining)", "correct picks (declining)"]
print("median profile of the top 50, split by whether the model was right:")
print(profile.to_string())
print()
print("The false alarms have median CTR 0.000 and median 0 clicks. A page with no clicks has no")
print("click trend that can decline -- it gets a maximum ctr_gap_ratio (the 1.000 cap) for having")
print("no clicks at all, and the model follows the rule's own blind spot straight into it.")
print("Same signature I found in ML-08, now reproduced on a different split. Named fix for v2:")
print("require a minimum click count in the eligibility gate, or score zero-click pages separately.")

honest split: model's top 50 -> 44 of 50 were really declining
test-set base rate for comparison: 0.551

median profile of the top 50, split by whether the model was right:
                                ctr  clicks_90d  impressions_90d  avg_position  days_since_last_update  ctr_gap_ratio
false alarms (not declining)  0.000         0.0           1209.0           3.2                    20.0          1.000
correct picks (declining)     0.035         1.0           1673.0           3.0                    20.0          0.834

The false alarms have median CTR 0.000 and median 0 clicks. A page with no clicks has no
click trend that can decline -- it gets a maximum ctr_gap_ratio (the 1.000 cap) for having
no clicks at all, and the model follows the rule's own blind spot straight into it.
Same signature I found in ML-08, now reproduced on a different split. Named fix for v2:
require a minimum click count in the eligibility gate, or score zero-click pages separately.


## 5. What changed, in one place

| | before this audit | after |
|---|---|---|
| Split | random rows | grouped by `client_id`, 0 clients shared |
| Headline ROC-AUC | 0.728 | **0.618** |
| Headline P@50 | 0.94 | **0.88** |
| Gap, over 8 draws | — | random higher **8 of 8**, mean +0.080 AUC |
| Leakage test | assumed clean | proven: adding `trend_pct` sends AUC to 1.000 |
| Claim | "the model beats the rule" | "ranked 44 of 50 correctly on unseen clients; directional, not decisive" |

**The one-line version.** My model got worse on paper and more useful in practice. The 0.728
was measuring memory of clients it had already met.

**What I would still not claim.** That this predicts decline before it happens — the label
shares its window with the features. That the 0.618 is precise — 9 test clients is thin. That
the model should replace the rule — it wins 6 of 8 splits, which is directional evidence for
running both and reviewing the union.

**What I would take to a content team:** a ranked list of pages to review first, with the
zero-click blind spot written on the label.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.